# UC CosMx: OGN+RSPO3+ Fib — SPP1+ Mac Spatial Interaction (Fig 1I)

1. NicheNet ligand-receptor prioritization (SPP1+ Mac → OGN+RSPO3+ Fib)
2. TNC downstream target spatial expression
3. Spatial LR validation

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged"
OUTPUT_DIR = DATA_DIR + "/fib_mac_interaction"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from scipy.spatial import cKDTree

In [ ]:
adata_wtx=sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')

In [ ]:
adata_wtx_orig = adata_wtx.copy()

In [ ]:
adata_wtx=adata_wtx[adata_wtx.obs['patient'].isin(['UC_P1','UC_P2'])]

In [ ]:
adata_wtx.obs.index = adata_wtx.obs['cell_id'].tolist()

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.cm as cm

def truncate_cmap(cmap, minval=0.12, maxval=1.0, n=256):
    return mcolors.LinearSegmentedColormap.from_list(
        "trunc", cmap(np.linspace(minval, maxval, n))
    )

reds_trunc = truncate_cmap(cm.Reds, 0.25, 1.0)


In [ ]:

adata1_slide3 = adata_wtx[adata_wtx.obs['slide'] == 'UC_batch1_slide3']
coords = adata1_slide3.obsm['spatial_fov']

ct_col = "aucell_cell_type_short"
cell_types = adata1_slide3.obs[ct_col].astype(str).fillna("NA")

# Define the exact labels you want
ogn_ct = "OGN+RSPO3+ Fib"
mac_ct = "Mac S+M+S+"   # <-- make sure this matches your obs labels
# if your data uses "Mac S+M+S+" but you want to label it as SELENOP+..., then set:
# mac_ct = "Mac S+M+S+"

ogn_mask = (cell_types.values == ogn_ct)
mac_mask = (cell_types.values == mac_ct)

def get_expr(gene):
    x = adata1_slide3[:, gene].X
    return x.toarray().ravel() if hasattr(x, "toarray") else np.asarray(x).ravel()

tnc = get_expr("TNC")
mmp1 = get_expr("MMP1")

fig = plt.figure(figsize=(9, 9))
gs = fig.add_gridspec(1, 3, width_ratios=[20, 1, 1], wspace=0.3)

ax  = fig.add_subplot(gs[0, 0])
cax1 = fig.add_subplot(gs[0, 1])
cax2 = fig.add_subplot(gs[0, 2])

# background
ax.scatter(coords[:, 0], coords[:, 1], c="#d0d0d0", s=3, linewidths=0)

# overlays
reds_trunc = truncate_cmap(cm.Reds, 0.12, 1.0)
tnc_vals = tnc[ogn_mask]
tnc_vmax = np.quantile(tnc_vals, 0.99) if len(tnc_vals) else 1.0
sc1 = ax.scatter(
    coords[ogn_mask, 0], coords[ogn_mask, 1],
    c=tnc_vals,
    s=18,
    linewidths=0,
    cmap=reds_trunc,
    vmin=np.quantile(tnc_vals, 0.05),
    vmax=tnc_vmax
)
blues_trunc = truncate_cmap(cm.Blues, 0.12, 1.0)
mmp1_vals = mmp1[mac_mask]
mmp1_vmax = np.quantile(mmp1_vals, 0.99) if len(mmp1_vals) else 1.0
sc2 = ax.scatter(coords[mac_mask, 0], coords[mac_mask, 1],
                 c=mmp1_vals, s=18, linewidths=0,
                     cmap=blues_trunc, vmin=np.quantile(mmp1_vals, 0.05),vmax=mmp1_vmax)

ax.set_title("TNC (OGN+RSPO3+ Fib) and MMP1 (SELENOP+MMP9+SPP1+ Mac)")
ax.set_xlabel("X coordinate")
ax.set_ylabel("Y coordinate")
ax.set_aspect("auto")
ax.grid(False)

# identical-height colorbars
cb1 = fig.colorbar(sc1, cax=cax1)
cb1.set_label("TNC expression (OGN+RSPO3+ Fib)")

cb2 = fig.colorbar(sc2, cax=cax2)
cb2.set_label("MMP1 expression (SELENOP+MMP9+SPP1+ Mac)")

plt.show()


In [ ]:

adata1_slide3 = adata_wtx[adata_wtx.obs['slide'] == 'UC_batch1_slide4']
coords = adata1_slide3.obsm['spatial_fov']

ct_col = "aucell_cell_type_short"
cell_types = adata1_slide3.obs[ct_col].astype(str).fillna("NA")

# Define the exact labels you want
ogn_ct = "OGN+RSPO3+ Fib"
mac_ct = "Mac S+M+S+"   # <-- make sure this matches your obs labels
# if your data uses "Mac S+M+S+" but you want to label it as SELENOP+..., then set:
# mac_ct = "Mac S+M+S+"

ogn_mask = (cell_types.values == ogn_ct)
mac_mask = (cell_types.values == mac_ct)

def get_expr(gene):
    x = adata1_slide3[:, gene].X
    return x.toarray().ravel() if hasattr(x, "toarray") else np.asarray(x).ravel()

tnc = get_expr("TNC")
mmp1 = get_expr("MMP1")

fig = plt.figure(figsize=(9, 9))
gs = fig.add_gridspec(1, 3, width_ratios=[20, 1, 1], wspace=0.3)

ax  = fig.add_subplot(gs[0, 0])
cax1 = fig.add_subplot(gs[0, 1])
cax2 = fig.add_subplot(gs[0, 2])

# background
ax.scatter(coords[:, 0], coords[:, 1], c="#d0d0d0", s=3, linewidths=0)

# overlays
reds_trunc = truncate_cmap(cm.Reds, 0.12, 1.0)
tnc_vals = tnc[ogn_mask]
tnc_vmax = np.quantile(tnc_vals, 0.99) if len(tnc_vals) else 1.0
sc1 = ax.scatter(
    coords[ogn_mask, 0], coords[ogn_mask, 1],
    c=tnc_vals,
    s=18,
    linewidths=0,
    cmap=reds_trunc,
    vmin=np.quantile(tnc_vals, 0.05),
    vmax=tnc_vmax
)
blues_trunc = truncate_cmap(cm.Blues, 0.12, 1.0)
mmp1_vals = mmp1[mac_mask]
mmp1_vmax = np.quantile(mmp1_vals, 0.99) if len(mmp1_vals) else 1.0
sc2 = ax.scatter(coords[mac_mask, 0], coords[mac_mask, 1],
                 c=mmp1_vals, s=18, linewidths=0,
                     cmap=blues_trunc, vmin=np.quantile(mmp1_vals, 0.05),vmax=mmp1_vmax)

ax.set_title("TNC (OGN+RSPO3+ Fib) and MMP1 (SELENOP+MMP9+SPP1+ Mac)")
ax.set_xlabel("X coordinate")
ax.set_ylabel("Y coordinate")
ax.set_aspect("auto")
ax.grid(False)

# identical-height colorbars
cb1 = fig.colorbar(sc1, cax=cax1)
cb1.set_label("TNC expression (OGN+RSPO3+ Fib)")

cb2 = fig.colorbar(sc2, cax=cax2)
cb2.set_label("MMP1 expression (SELENOP+MMP9+SPP1+ Mac)")

plt.show()


## nichenet


## Example with TGFB1:

### **Complete pathway:**
```
SPP1+ Macrophage
    |
    | Expresses/secretes: TGFB1 (ligand)
    ↓
    [Extracellular space]
    ↓
OGN+RSPO3+ Fibroblast
    |
    | Expresses: TGFBR1 + TGFBR2 (receptors)
    ↓
    | TGFB1 binds to TGFBR1/TGFBR2
    ↓
    | Intracellular signaling cascade
    ↓
    | Activates: SMAD2/SMAD3 (TFs)
    ↓
    | SMAD2/3 enter nucleus
    ↓
    | Turn on target genes: COL1A1, ACTA2, FN1, etc.

In [ ]:
# Load NicheNet results
ligand_tf_predictions = pd.read_csv("/path/to/scrna/uc/ogn_spp1_mac_nichenet_ligand_tf_predictions.csv")
predicted_tfs = pd.read_csv("/path/to/scrna/uc/ogn_spp1_mac_nichenet_predicted_tfs.csv")
tf_target_network = pd.read_csv("/path/to/scrna/uc/ogn_spp1_mac_nichenet_tf_target_network.csv")
ligand_activities = pd.read_csv("/path/to/scrna/uc/ogn_spp1_mac_nichenet_ligand_activities.csv")

In [ ]:
# Calculate proximity (if not already done)
ogn_fib_mask = adata_wtx.obs['aucell_cell_type_short'] == 'OGN+RSPO3+ Fib'
spp1_mac_mask = adata_wtx.obs['aucell_cell_type_short'] == 'Mac S+M+S+'


In [ ]:
all_ogn_coords = adata_wtx[ogn_fib_mask].obsm['spatial_fov']
all_spp1_coords = adata_wtx[spp1_mac_mask].obsm['spatial_fov']
# Get coordinates

In [ ]:
tree = cKDTree(all_spp1_coords)
distances, _ = tree.query(all_ogn_coords, k=1)

In [ ]:
print(f"\nDistance statistics:")
print(f"  Min: {distances.min():.2f}")
print(f"  Max: {distances.max():.2f}")
print(f"  Mean: {distances.mean():.2f}")
print(f"  Median: {np.median(distances):.2f}")


In [ ]:
# Create OGN+ fibroblast subset with distance info
adata_ogn = adata_wtx[ogn_fib_mask].copy()
adata_ogn.obs['distance_to_spp1_mac'] = distances

In [ ]:
# Test TF expression near vs far
candidate_tfs = predicted_tfs['tf'].tolist()

In [ ]:

tf_targets = (
    tf_target_network.groupby("from")["to"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
    .rename(columns={"from": "TF", "to": "targets"})
)


In [ ]:
top_ligands = ligand_activities.nlargest(50, 'auroc')
top_ligands

In [ ]:
import pyreadr

In [ ]:
result = pyreadr.read_r(
    "/path/to/scrna/uc/nichenet_tf_target_matrix.rds"
)

tf_target_matrix = list(result.values())[0]

In [ ]:
result = pyreadr.read_r(
    "/path/to/nichenet/ligand_target_matrix_nsga2r_final.rds"
)

ligand_target_matrix = list(result.values())[0]

In [ ]:

result = pyreadr.read_r(
    "/path/to/nichenet/lr_network_human_21122021.rds"
)

lr_network = list(result.values())[0]

In [ ]:
# Check ligand expression in SPP1+ macs
spp1_mac_mask = adata_wtx.obs['aucell_cell_type_short'] == 'Mac S+M+S+'

ligand_in_macs = []

for ligand in top_ligands['test_ligand'].tolist():
    if ligand in adata_wtx.var_names:
        ligand_expr = adata_wtx[spp1_mac_mask, ligand].X
        if hasattr(ligand_expr, 'toarray'):
            ligand_expr = ligand_expr.toarray().flatten()
        
        mean_expr = ligand_expr.mean()
        pct_expr = (ligand_expr > 0).sum() / len(ligand_expr) * 100
        auroc = ligand_activities[ligand_activities['test_ligand'] == ligand]['auroc'].values[0]
        
        ligand_in_macs.append({
            'ligand': ligand,
            'auroc': auroc,
            'mean_expr_in_macs': mean_expr,
            'pct_macs_expressing': pct_expr
        })

ligand_macs_df = pd.DataFrame(ligand_in_macs)
ligand_macs_df = ligand_macs_df.sort_values('auroc', ascending=False)



In [ ]:
validated_ligands=ligand_macs_df

In [ ]:
# For validated ligands, get their downstream TFs
validated_ligand_tfs = ligand_tf_predictions[
    ligand_tf_predictions['ligand'].isin(validated_ligands['ligand'])
].copy()

# Merge with TF expression/activity results
validated_ligand_tfs = validated_ligand_tfs.merge(
    combined[['TF', 'log2FC', 'pvalue_expr', 'activity_diff', 'pvalue_activity']],
    left_on='tf',
    right_on='TF',
    how='left'
)

validated_ligand_tfs = validated_ligand_tfs.sort_values(['score'], ascending=[ False])

print("\n=== TFs Downstream of Validated Ligands ===")
validated_ligand_tfs[['ligand', 'tf', 'score', 'log2FC', 'pvalue_expr', 'activity_diff', 'pvalue_activity']].head(50)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Top ligands from your analysis
top_ligands = validated_ligands['ligand'].tolist()
#['MMP14', 'MMP1', 'COL3A1', 'C1QB', 'S100A4', 'APOC2', 'CTSD']

# Load spatial data
# adata_spatial should have both SPP1+ macs and OGN+ fibs

ligand_target_correlations = []

for ligand in top_ligands:
    # Check if ligand is in dataset
    if ligand not in adata_wtx.var_names:
        print(f"Skipping {ligand} - not in dataset")
        continue
    
    # Get TFs downstream of this ligand
    ligand_tfs = combined[combined['upstream_ligands'].str.contains(ligand, na=False)]['TF'].tolist()
    
    # For each tissue
    for slide_id in adata_wtx.obs['slide'].unique():
        slide_mask = adata_wtx.obs['slide'] == slide_id
        slide_data = adata_wtx[slide_mask]
        
        # Get cell masks
        ogn_mask = slide_data.obs['aucell_cell_type_short'] == 'OGN+RSPO3+ Fib'
        spp1_mask = slide_data.obs['aucell_cell_type_short'] == 'Mac S+M+S+'
        
        if ogn_mask.sum() < 10 or spp1_mask.sum() < 5:
            continue
        
        # Get coordinates
        ogn_coords = slide_data[ogn_mask].obsm['spatial_fov']
        spp1_coords = slide_data[spp1_mask].obsm['spatial_fov']
        
        # Get ligand expression in SPP1+ macs
        ligand_expr_macs = slide_data[spp1_mask, ligand].X
        if hasattr(ligand_expr_macs, 'toarray'):
            ligand_expr_macs = ligand_expr_macs.toarray().flatten()
        
        # For each OGN+ fib, find nearest SPP1+ mac
        tree = cKDTree(spp1_coords)
        distances, nearest_mac_indices = tree.query(ogn_coords, k=1)
        
        # Get ligand level in nearest mac for each fib
        ligand_in_nearest_mac = ligand_expr_macs[nearest_mac_indices]
        
        # Test correlation with downstream TFs in fibroblasts
        for tf in ligand_tfs:
            if tf not in slide_data.var_names:
                continue
            
            tf_expr_fibs = slide_data[ogn_mask, tf].X
            if hasattr(tf_expr_fibs, 'toarray'):
                tf_expr_fibs = tf_expr_fibs.toarray().flatten()
            
            # Correlation: ligand in nearest mac vs TF in fib
            if len(np.unique(ligand_in_nearest_mac)) > 1:  # Need variation
                corr, pval = stats.spearmanr(ligand_in_nearest_mac, tf_expr_fibs)
                
                ligand_target_correlations.append({
                    'slide': slide_id,
                    'ligand': ligand,
                    'TF': tf,
                    'correlation': corr,
                    'pvalue': pval,
                    'n_fibs': len(tf_expr_fibs),
                    'n_macs': len(ligand_expr_macs),
                    'mean_distance': distances.mean()
                })

# Combine results
corr_df = pd.DataFrame(ligand_target_correlations)

if len(corr_df) > 0:
    # FDR correction
    corr_df['padj'] = stats.false_discovery_control(corr_df['pvalue'])
    
    # Aggregate across tissue
    corr_summary = corr_df.groupby(['ligand', 'TF']).agg({
        'correlation': 'mean',
        'pvalue': lambda x: stats.combine_pvalues(x, method='fisher')[1],
        'n_fibs': 'sum'
    }).reset_index()
    
    corr_summary['padj'] = stats.false_discovery_control(corr_summary['pvalue'])
    corr_summary = corr_summary.sort_values('pvalue')
    
    print("\n=== Ligand-TF Spatial Correlations ===")
    print(corr_summary.head(20))
    
    #corr_summary.to_csv('ligand_tf_spatial_correlations.csv', index=False)

In [ ]:
ligand_df = ligand_activities.sort_values('auroc', ascending=False)[:20]
ligand_df

In [ ]:
ligand_df=ligand_df.merge(validated_ligands, how = 'left', left_on = 'test_ligand', right_on = 'ligand')


In [ ]:
ligand_df = ligand_df.dropna(subset=['mean_expr_in_macs', 'pct_macs_expressing'])
ligand_df

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================
# FIGURE - LONG + FLAT, VERTICAL "STAND-UP" LOLLIPOPS
# ============================================

# Long + flat (wide, short). Increase width if you have lots of ligands.
fig, ax = plt.subplots(figsize=(8.7, 4.5))

# x positions for each ligand
x = list(range(len(ligand_df)))

# Colors: red for validated, light blue for others
stem_colors = [
    '#E63946' if lig in validated else '#A8DADC'
    for lig in ligand_df['test_ligand']
]

# Baseline (like your 0.5 reference, but now it's horizontal)
baseline = 0.5

# ---- stems (vertical) ----
for i, (idx, row) in enumerate(ligand_df.iterrows()):
    ax.plot([i, i], [baseline, row['auroc_x']],
            color=stem_colors[i],
            linewidth=4,
            alpha=0.85,
            solid_capstyle='round',
            zorder=2)

# ---- heads (dots) ----
ax.scatter(
    x,
    ligand_df['auroc_x'],
    s=150,
    c=stem_colors,
    edgecolors='black',
    linewidths=1,
    zorder=3
)

# ---- baseline reference line ----
ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5, linewidth=1.5, zorder=0)

# ============================================
# AXES / LABELS
# ============================================

ax.set_xticks(x)
ax.set_xticklabels(ligand_df['test_ligand'], rotation=45, ha='right', fontsize=10)

ax.set_xlabel('Mac S+M+S+ Ligands', fontsize=12)
ax.set_ylabel('AUROC for OGN+RSPO3+ Fib DEGs', fontsize=12)
ax.set_title('')

# Limits: give headroom above max and a little below baseline
ymax = float(ligand_df['auroc_x'].max())
ax.set_ylim([baseline - 0.02, ymax + 0.05])

# Clean up spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ============================================
# % EXPRESSING ANNOTATIONS (as a "column" above each lollipop)
# ============================================

# Put the % labels just above the dot, and keep them from colliding too much
for i, (idx, row) in enumerate(ligand_df.iterrows()):
    ax.text(
        i,
        row['auroc_x'] + 0.012,          # vertical offset above dot
        f"{row['pct_macs_expressing']:.1f}%",
        ha='center',
        va='bottom',
        fontsize=9,
        alpha=0.75,
        zorder=4
    )

# Small header for the % labels (top-left-ish of plotting area)
ax.text(
    -0.6,
    ymax + 0.045,
    "% Mac S+M+S+ Expressing",
    ha='left',
    va='top',
    fontsize=10,
    alpha=0.85
)

# ============================================
# LEGEND
# ============================================

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#E63946', markersize=10,
           markeredgecolor='black', markeredgewidth=1,
           label='Spatially Significant'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#A8DADC', markersize=10,
           markeredgecolor='black', markeredgewidth=1,
           label='Predicted')
]

ax.legend(
    handles=legend_elements,
    loc='upper left',
    bbox_to_anchor=(0.77, 1.0),
    fontsize=9.5,
    frameon=True,
    borderaxespad=0
)

plt.tight_layout()
plt.show()


In [ ]:
corr_ligands = corr_summary[:19]['ligand'].unique().tolist()
corr_ligand

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(7,6))

spatial_df = corr_summary.sort_values('pvalue', ascending=True).head(19)

labels = [f"{row['ligand']} → {row['TF']}" for _, row in spatial_df.iterrows()]

colors = [
    '#D62828' if row['pvalue'] < 0.05
    else '#F77F00' if row['pvalue'] < 0.1
    else '#FCBF49'
    for _, row in spatial_df.iterrows()
]

ax.barh(range(len(spatial_df)), spatial_df['correlation'],
        color=colors, edgecolor='black', linewidth=1.2)

ax.set_yticks(range(len(spatial_df)))
ax.set_yticklabels(labels, fontsize=11)
ax.set_ylabel('Ligand (Mac S+M+S+) → TF (OGN+RSPO3+ Fib)', fontsize=13)
ax.set_xlabel('Spatial Correlation (r) of Ligand and TF', fontsize=13)
ax.set_title('Spatial Validation of Ligand → TF Eaxpression', fontsize=14, loc='left')

ax.axvline(0, color='black', linewidth=1)
ax.set_xlim(-0.15, 0.15)
ax.invert_yaxis()

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# annotate correlation values
for i, (_, row) in enumerate(spatial_df.iterrows()):
    x_pos = row['correlation'] + 0.004 if row['correlation'] > 0 else row['correlation'] - 0.004
    ha = 'left' if row['correlation'] > 0 else 'right'
    ax.text(x_pos, i, f"r:{row['correlation']:.3f}",
            va='center', ha=ha, fontsize=10)

# legend
legend_elements = [
    Patch(facecolor='#D62828', edgecolor='black', label='p < 0.05'),
    Patch(facecolor='#F77F00', edgecolor='black', label='p < 0.1'),
    Patch(facecolor='#FCBF49', edgecolor='black', label='p < 0.2')
]

ax.legend(handles=legend_elements, title="p-value", frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
corr_ligands_receptor =lr_network[lr_network['from'].isin(corr_ligands)][['from','to']]
corr_ligands_receptor=corr_ligands_receptor.groupby('from')['to'].apply(list).reset_index()
corr_ligands_receptor.to_csv('corr_ligands_receptors.csv')

In [ ]:
tf_target_matrix[['TNC']].sort_values('TNC',ascending=False)[:20]

In [ ]:
ligand_target_matrix[['MMP1']].sort_values('MMP1',ascending=False)[:50]

In [ ]:
# Filter for TNC only
tnc_targets = tf_target_matrix[['TNC']].sort_values('TNC', ascending=False)[:12]

print(f"TNC regulates {len(tnc_targets)} genes")
print(tnc_targets.head(12))